# Lab C: Topic modeling on book titles

**Decision log**

| Choice | What we did | Why |
|---|---|---|
| Duplicates | removed (142) | repeated titles count twice |
| Train/test split | none | no labels, so nothing to test |
| Cleaning | lowercase, remove 5th/3rd/4e | edition numbers are not subjects |
| Stop words | sklearn list + noise list | words like edition, card, package are not subjects |
| min_df | 5 | rarer words show no pattern |
| LDA features | CountVectorizer | LDA works on counts |
| NMF features | TfidfVectorizer | lowers weight of common words |
| random_state | 2 | same results every run |
| Number of topics | 10 | clearest topics (see step 6) |

## Step 1: Explore

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation, NMF

random_state = 2
np.random.seed(random_state)

with open("data/all_book_titles.txt", encoding="utf-8") as f:
    titles = [t.strip() for t in f.read().splitlines() if t.strip()]

df = pd.DataFrame({"title": titles})
print("titles:", len(df))
print("duplicates:", len(df) - df["title"].nunique())

df["n_words"] = df["title"].str.split().str.len()
print(df["n_words"].describe())

print(df["title"].str.lower().str.split().explode().value_counts().head(40))

titles: 2373
duplicates: 142
count    2373.000000
mean        7.142857
std         4.096361
min         1.000000
25%         4.000000
50%         6.000000
75%         9.000000
max        36.000000
Name: n_words, dtype: float64
title
and             932
of              576
the             489
edition)        321
to              320
a               304
in              304
for             258
introduction    236
with            180
an              175
edition         136
:               130
history         104
&               104
science         100
computer         88
approach         87
access           77
business         75
human            71
applications     68
principles       67
analysis         64
essentials       63
series           62
biology          61
art              58
theory           58
guide            56
fundamentals     56
statistics       53
card             53
chemistry        52
concepts         51
anatomy          51
design           50
data             50
world  

There are 2373 titles and 142 are duplicates. The median title is 6 words long. The most common words are stop words plus publisher words like edition, introduction, access, series and card.

Noise words: edition, introduction, approach, access, card, plus, package, series, volume, book, brief, essentials, etext, student, manual, myartslab, masteringa, masteringbiology, masteringchemistry, and edition numbers like 5th and 3rd.

## Step 2: Clean and vectorize

In [2]:
df = df.drop_duplicates(subset="title").reset_index(drop=True)
print("titles left:", len(df))

def prep(text):
    text = text.lower()
    text = re.sub(r"\b\d+(st|nd|rd|th)\b", " ", text)
    text = re.sub(r"\b\d+e\b", " ", text)
    return text

noise = ["edition", "introduction", "approach", "access", "card", "plus", "package",
         "series", "volume", "book", "brief", "essentials", "etext", "student", "manual",
         "myartslab", "masteringa", "masteringbiology", "masteringchemistry"]
stop_list = list(ENGLISH_STOP_WORDS) + noise

count_vec = CountVectorizer(stop_words=stop_list, preprocessor=prep, min_df=5)
X_counts = count_vec.fit_transform(df["title"])

tfidf_vec = TfidfVectorizer(stop_words=stop_list, preprocessor=prep, min_df=5)
X_tfidf = tfidf_vec.fit_transform(df["title"])

print(X_counts.shape, X_tfidf.shape)

titles left: 2231
(2231, 415) (2231, 415)


LDA gets counts because it is a probability model built on how many times words show up. NMF can use tf-idf because it is just a matrix factorization.

## Step 3: Fit both models (k = 10, chosen in step 6)

In [3]:
K = 10
lda = LatentDirichletAllocation(n_components=K, random_state=random_state)
lda.fit(X_counts)

nmf = NMF(n_components=K, random_state=random_state)
nmf.fit(X_tfidf)

,"n_components n_components: int or {'auto'} or None, default='auto'Number of components. If `None`, all features are kept.If `n_components='auto'`, the number of components is automatically inferredfrom W or H shapes... versionchanged:: 1.4 Added `'auto'` value... versionchanged:: 1.6 Default value changed from `None` to `'auto'`.",10
,"random_state random_state: int, RandomState instance or None, default=NoneUsed for initialisation (when ``init`` == 'nndsvdar' or'random'), and in Coordinate Descent. Pass an int for reproducibleresults across multiple function calls.See :term:`Glossary <random_state>`.",2
,"init init: {'random', 'nndsvd', 'nndsvda', 'nndsvdar', 'custom'}, default=NoneMethod used to initialize the procedure.Valid options:- `None`: 'nndsvda' if n_components <= min(n_samples, n_features), otherwise random.- `'random'`: non-negative random matrices, scaled with: `sqrt(X.mean() / n_components)`- `'nndsvd'`: Nonnegative Double Singular Value Decomposition (NNDSVD) initialization (better for sparseness)- `'nndsvda'`: NNDSVD with zeros filled with the average of X (better when sparsity is not desired)- `'nndsvdar'` NNDSVD with zeros filled with small random values (generally faster, less accurate alternative to NNDSVDa for when sparsity is not desired)- `'custom'`: Use custom matrices `W` and `H` which must both be provided... versionchanged:: 1.1 When `init=None` and n_components is less than n_samples and n_features defaults to `nndsvda` instead of `nndsvd`.",None
,"solver solver: {'cd', 'mu'}, default='cd'Numerical solver to use:- 'cd' is a Coordinate Descent solver.- 'mu' is a Multiplicative Update solver... versionadded:: 0.17 Coordinate Descent solver... versionadded:: 0.19 Multiplicative Update solver.",'cd'
,"beta_loss beta_loss: float or {'frobenius', 'kullback-leibler', 'itakura-saito'}, default='frobenius'Beta divergence to be minimized, measuring the distance between Xand the dot product WH. Note that values different from 'frobenius'(or 2) and 'kullback-leibler' (or 1) lead to significantly slowerfits. Note that for beta_loss <= 0 (or 'itakura-saito'), the inputmatrix X cannot contain zeros. Used only in 'mu' solver... versionadded:: 0.19",'frobenius'
,"tol tol: float, default=1e-4Tolerance of the stopping condition.",0.0001
,"max_iter max_iter: int, default=200Maximum number of iterations before timing out.",200
,"alpha_W alpha_W: float, default=0.0Constant that multiplies the regularization terms of `W`. Set it to zero(default) to have no regularization on `W`... versionadded:: 1.0",0.0
,"alpha_H alpha_H: float or ""same"", default=""same""Constant that multiplies the regularization terms of `H`. Set it to zero tohave no regularization on `H`. If ""same"" (default), it takes the same value as`alpha_W`... versionadded:: 1.0",'same'
,"l1_ratio l1_ratio: float, default=0.0The regularization mixing parameter, with 0 <= l1_ratio <= 1.For l1_ratio = 0 the penalty is an elementwise L2 penalty(aka Frobenius Norm).For l1_ratio = 1 it is an elementwise L1 penalty.For 0 < l1_ratio < 1, the penalty is a combination of L1 and L2... versionadded:: 0.17 Regularization parameter *l1_ratio* used in the Coordinate Descent solver.",0.0
,"verbose verbose: int, default=0Whether to be verbose.",0


## Step 4: Top 10 words

In [4]:
def show_topics(model, vectorizer, n=10):
    words = vectorizer.get_feature_names_out()
    for i, comp in enumerate(model.components_):
        top = [words[j] for j in comp.argsort()[-n:][::-1]]
        print("Topic", i, ":", ", ".join(top))

print("LDA")
show_topics(lda, count_vec)
print()
print("NMF")
show_topics(nmf, tfidf_vec)

LDA
Topic 0 : computer, economics, physiology, design, anatomy, data, systems, java, principles, structures
Topic 1 : probability, calculus, applications, nutrition, microbiology, geography, foundations, gender, immunology, concepts
Topic 2 : history, theory, art, philosophy, america, latin, film, american, global, game
Topic 3 : human, learning, machine, east, pharmacology, sexuality, asian, feminism, criminology, diaspora
Topic 4 : biology, business, world, molecular, law, concepts, health, music, hinduism, life
Topic 5 : chemistry, language, physical, science, management, natural, processing, organic, database, fundamentals
Topic 6 : understanding, judaism, differential, equations, islam, problems, guide, study, mechanics, pathophysiology
Topic 7 : psychology, sociology, marketing, engineering, earth, science, printed, thermodynamics, astronomy, ecology
Topic 8 : finance, real, statistics, physics, analysis, mechanics, engineers, quantum, medicine, basic
Topic 9 : science, literatur

In [5]:
# labels (made after reading the top titles in step 5)
nmf_labels = {
    0: "History, art and culture",
    1: "Computer systems and networks",
    2: "Human anatomy and physiology",
    3: "Biology",
    4: "Business, statistics and economics",
    5: "Chemistry",
    6: "Earth and political science (mixed)",
    7: "Psychology",
    8: "Sociology and social science",
    9: "Math, theory and analysis (broad)",
}

lda_labels = {
    0: "Computer hardware and data structures",
    1: "Unclear",
    2: "Art history",
    3: "Unclear",
    4: "Business law",
    5: "Natural language processing",
    6: "Differential equations",
    7: "Unclear",
    8: "Corporate finance",
    9: "Unclear",
}

Unclear topics: LDA 1 (immunology, applied calculus, a farm bill hearing), LDA 3 (ethics, machine learning, pharmacology, women and politics), LDA 7 (only shares the Printed Access Card text), LDA 9 (literature anthologies and actuarial books). NMF 6 mixes earth science and political science because both have the word science. NMF 9 is a catch-all.

## Step 5: Top 5 titles per topic

In [6]:
nmf_scores = nmf.transform(X_tfidf)
lda_scores = lda.transform(X_counts)

def show_top_titles(scores, labels, name):
    for topic in range(scores.shape[1]):
        print(name, "topic", topic, "-", labels[topic])
        for pos in scores[:, topic].argsort()[-5:][::-1]:
            print("   ", df["title"].iloc[pos][:95])
        print()

show_top_titles(nmf_scores, nmf_labels, "NMF")
show_top_titles(lda_scores, lda_labels, "LDA")

NMF topic 0 - History, art and culture
    Art History, Volume 1 (4th Edition) (MyArtsLab Series)
    Art History, Volume 2 (4th Edition) (MyArtsLab Series)
    Art History, Combined (4th Edition) (MyArtsLab Series)
    Art History Volume 2 (5th Edition)
    Art History (5th Edition)

NMF topic 1 - Computer systems and networks
    Computer Forensics and Cyber Crime: An Introduction (3rd Edition)
    Computer Organization
    Structured Computer Organization
    Structured Computer Organization (6th Edition)
    Computer Organization and Architecture (9th Edition) (William Stallings Books on Computer and D

NMF topic 2 - Human anatomy and physiology
    Human Anatomy and Physiology
    Hole's Human Anatomy & Physiology
    Essentials of Human Anatomy & Physiology (10th Edition)
    Hole's Essentials of Human Anatomy & Physiology
    Human Anatomy & Physiology (9th Edition)

NMF topic 3 - Biology
    Essentials of Biology
    Biology
    Brock Biology of Microorganisms (13th Edition)
  

NMF titles matched our labels for all 10 topics. LDA matched for 5 topics (art history, differential equations, NLP, corporate finance, business law) and the other 4 were unclear. LDA works from raw counts, so groups of near identical titles with long repeated series names pull a whole topic toward them. Many NMF titles tie on score (all the Art History editions look the same after cleaning), so the five shown can change a bit.

## Step 6: Number of topics

In [7]:
def top_words(model, vectorizer, n=8):
    words = vectorizer.get_feature_names_out()
    return [", ".join(words[j] for j in comp.argsort()[-n:][::-1]) for comp in model.components_]

results = []
for k in [5, 10, 15]:
    lda_k = LatentDirichletAllocation(n_components=k, random_state=random_state).fit(X_counts)
    nmf_k = NMF(n_components=k, random_state=random_state).fit(X_tfidf)
    results.append({"k": k, "NMF error": nmf_k.reconstruction_err_, "LDA perplexity": lda_k.perplexity(X_counts)})
    print("k =", k)
    print("LDA")
    for i, w in enumerate(top_words(lda_k, count_vec)):
        print("  ", i, w)
    print("NMF")
    for i, w in enumerate(top_words(nmf_k, tfidf_vec)):
        print("  ", i, w)
    print()

pd.DataFrame(results).round(2)

k = 5
LDA
   0 computer, design, systems, economics, physiology, data, anatomy, principles
   1 statistics, science, probability, analysis, language, calculus, natural, applications
   2 history, theory, art, finance, global, physical, earth, american
   3 chemistry, science, human, political, learning, organic, east, machine
   4 biology, world, health, sociology, psychology, molecular, guide, understanding
NMF
   0 history, art, world, american, america, film, new, modern
   1 science, computer, earth, political, organization, networks, physical, architecture
   2 human, anatomy, physiology, sexuality, anthropology, laboratory, principles, version
   3 biology, molecular, chemistry, concepts, campbell, life, organic, biochemistry
   4 business, statistics, economics, theory, analysis, probability, applications, principles



k = 10
LDA
   0 computer, economics, physiology, design, anatomy, data, systems, java
   1 probability, calculus, applications, nutrition, microbiology, geography, foundations, gender
   2 history, theory, art, philosophy, america, latin, film, american
   3 human, learning, machine, east, pharmacology, sexuality, asian, feminism
   4 biology, business, world, molecular, law, concepts, health, music
   5 chemistry, language, physical, science, management, natural, processing, organic
   6 understanding, judaism, differential, equations, islam, problems, guide, study
   7 psychology, sociology, marketing, engineering, earth, science, printed, thermodynamics
   8 finance, real, statistics, physics, analysis, mechanics, engineers, quantum
   9 science, literature, theater, anthropology, new, bioinformatics, sex, reader
NMF
   0 history, art, american, world, america, film, new, modern
   1 computer, organization, networks, architecture, security, networking, design, systems
   2 human, an

k = 15
LDA
   0 physiology, economics, anatomy, human, data, principles, structures, java
   1 social, judaism, gender, life, film, nutrition, electronic, critical
   2 history, art, earth, modern, american, latin, america, science
   3 learning, accounting, machine, sexuality, astronomy, algorithms, human, women
   4 biology, theory, molecular, game, world, bioinformatics, ecology, immunology
   5 science, language, engineering, natural, political, processing, physical, thermodynamics
   6 methods, equations, differential, understanding, forestry, problems, criminology, exploring
   7 sociology, philosophy, practice, law, readings, buddhism, greek, reader
   8 finance, physics, mechanics, quantum, insurance, real, engineers, aerodynamics
   9 anthropology, theater, literature, new, anthology, evolution, cultural, sex
   10 world, marketing, psychology, global, east, asian, geography, contemporary
   11 chemistry, general, concepts, organic, relativity, biochemistry, biological, ethics

,k,NMF error,LDA perplexity
0,5,45.36,349.58
1,10,44.07,341.41
2,15,42.90,333.27


At k = 5 the topics were mixed (NMF put biology and chemistry together). At k = 10 NMF split out biology, chemistry, psychology, sociology and anatomy, and LDA found art history, differential equations, NLP and finance. At k = 15 NMF found some new subjects (statistics, physics, marketing) but also some messy topics, and LDA got worse.

NMF error went 45.36, 44.07, 42.90 and LDA perplexity went 349.58, 341.41, 333.27. Both keep going down as k grows because more topics always fit the training data better, so they can't pick k for us. The perplexity is also on the training data. We picked k = 10 because it gave the clearest topics for both models.

## Step 7: Topic mixtures for five titles

In [8]:
def show_mixture(pos, scores, labels, name):
    row = scores[pos] / (scores[pos].sum() + 1e-9)
    order = row.argsort()[::-1][:3]
    print("   ", name, ":", "; ".join(f"T{t} {labels[t]} {row[t]:.0%}" for t in order))

for pos in [0, 100, 500, 1000, 2000]:
    print(df["title"].iloc[pos][:110])
    show_mixture(pos, nmf_scores, nmf_labels, "NMF")
    show_mixture(pos, lda_scores, lda_labels, "LDA")
    print()

Philosophy of Sex and Love A Reader
    NMF : T8 Sociology and social science 43%; T0 History, art and culture 22%; T9 Math, theory and analysis (broad) 16%
    LDA : T9 Unclear 52%; T2 Art history 27%; T1 Unclear 3%

The Theater Experience
    NMF : T0 History, art and culture 36%; T8 Sociology and social science 26%; T7 Psychology 23%
    LDA : T9 Unclear 37%; T6 Differential equations 37%; T7 Unclear 3%

The Convergence of Race, Ethnicity, and Gender: Multiple Identities in Counseling (4th Edition) (Merrill Couns
    NMF : T8 Sociology and social science 66%; T7 Psychology 28%; T0 History, art and culture 3%
    LDA : T1 Unclear 77%; T9 Unclear 3%; T6 Differential equations 3%

Essentials of Human Anatomy & Physiology (10th Edition)
    NMF : T2 Human anatomy and physiology 100%; T9 Math, theory and analysis (broad) 0%; T8 Sociology and social science 0%
    LDA : T0 Computer hardware and data structures 77%; T3 Unclear 3%; T6 Differential equations 3%

Essentials of Pharmacology fo

Essentials of Human Anatomy & Physiology is about one thing (NMF gives 100% to anatomy). The Convergence of Race, Ethnicity, and Gender is about two (sociology 66%, psychology 28%). The Theater Experience only has two words left after cleaning so its mixture is mostly a guess. Pharmacology has no good NMF topic at k = 10. Specific technical titles land on one topic and short or broad ones spread out, but a mixed result can also just mean the model has no good topic for that title.

## Investigations

(to add)